# Agent 12 — Human Review Layer

**What this notebook does:**  
Records every decision where a human team member overrides the quantitative model — whether adding a stock the algorithm ranked low, removing one the algorithm ranked high, or adjusting a weight. Creates an auditable decision log required by the mandate.

**How to present this to investors:**  
> *Our human review layer ensures the pipeline is governed, not just automated. Every deviation from the quantitative model is logged here with a written rationale, the name of the decision-maker, and the date. The panel can inspect any override and challenge the reasoning. This is how we demonstrate that the AI is a tool supporting human judgement, not a black box replacing it.*

**Why this matters for the Q&A:**  
The assignment requires at least 3 concrete human override examples. This notebook is the evidence.  
The panel will specifically ask: *"Give me an example where you overruled the model and why."*

**Run after:** `04_portfolio_construction.ipynb` and `09_greenwashing.ipynb`

## Step 1 — Load the current portfolio and scoring data

In [1]:
import pandas as pd
import numpy as np
import os
import json
import glob
from datetime import date

TODAY = str(date.today())

# Load the latest portfolio
port_files = sorted(glob.glob("../outputs/portfolio/final_portfolio_*.csv"))
if port_files:
    portfolio = pd.read_csv(port_files[-1])
    print(f"Portfolio loaded: {port_files[-1]}")
    print(f"Holdings: {len(portfolio)}")
    print(portfolio.head())
else:
    portfolio = pd.DataFrame()
    print("No final portfolio found. Run 04_portfolio_construction.ipynb first.")
    print("You can still log overrides now — they will apply when the portfolio is built.")

Portfolio loaded: ../outputs/portfolio\final_portfolio_2026-05-06.csv
Holdings: 20
      ticker  E_score  S_score  G_score  ESG_score data_vintage  \
0    ASML.AS    76.71    51.43    34.54      56.48   2026-05-06   
1    INGA.AS    77.65    40.97    38.98      55.04   2026-05-06   
2  HEXA-B.ST    61.66    53.11    62.02      59.20   2026-05-06   
3     ITX.MC    89.88    61.78    56.82      71.53   2026-05-06   
4     ULVR.L    78.18    55.29    60.66      66.06   2026-05-06   

   annual_return_pct  annual_volatility_pct  sharpe_ratio  max_drawdown_pct  \
0              56.51                  28.06         1.907            -23.20   
1              41.44                  23.23         1.655            -21.07   
2              40.55                  24.93         1.506            -26.49   
3              25.63                  21.57         1.049            -27.06   
4              23.22                  16.93         1.194            -21.22   

  data_vintage_fin  sharpe_score  compo

## Step 2 — Add human override decisions

Edit the `OVERRIDES` list below. Add one entry for each decision that deviates from the quantitative model output.  
The assignment requires **at least 3 override examples**.

**Override types:**  
- `ADD` — Add a stock the model ranked below the threshold  
- `REMOVE` — Remove a stock the model would have included  
- `ADJUST_WEIGHT` — Change a holding's weight from the model's suggestion  
- `WATCHLIST_OVERRIDE` — Include a watchlisted stock despite the flag  
- `EXCLUSION_OVERRIDE` — Include an excluded stock (requires strong justification)

In [2]:
# ============================================================
#  TEAM ACTION REQUIRED — add your override decisions here
# ============================================================

OVERRIDES = [
    {
        "date": TODAY,
        "decided_by": "Captain",                          # team member name or role
        "ticker": "EXAMPLE1",
        "company_name": "Example Company 1",
        "override_type": "ADD",                           # ADD / REMOVE / ADJUST_WEIGHT / WATCHLIST_OVERRIDE
        "model_decision": "Ranked 24th — just outside top 20",
        "human_decision": "Include at 4.0% weight",
        "rationale": (
            "Recent CSRD disclosure (Feb 2025) shows significant improvement in Scope 3 reporting "
            "not yet reflected in the ESG dataset vintage. Company filed SBTi target in March 2025. "
            "Including this position improves sector diversification into Healthcare."
        ),
        "evidence": "Sustainability Report 2024, p.22; SBTi registry confirmation email 15 Mar 2025",
        "approved_by": "Team consensus"
    },
    {
        "date": TODAY,
        "decided_by": "ESG Specialist",
        "ticker": "EXAMPLE2",
        "company_name": "Example Company 2",
        "override_type": "REMOVE",
        "model_decision": "Ranked 8th — would be included at 5.2% weight",
        "human_decision": "Exclude from portfolio",
        "rationale": (
            "Greenwashing 8-Test flagged HIGH on Verification and Consistency dimensions. "
            "A May 2024 NGO report (ClientEarth) documents a €2.3 billion capex commitment "
            "to new LNG infrastructure directly contradicting the stated 2035 net zero target. "
            "Although the model scored this as borderline watchlist (2 HIGH flags), the RAG "
            "Operator verified both flags against primary sources — we apply a stricter interpretation."
        ),
        "evidence": "ClientEarth report May 2024, p.7; Annual Report 2024 capex table p.91",
        "approved_by": "Team consensus"
    },
    {
        "date": TODAY,
        "decided_by": "Data Engineer",
        "ticker": "EXAMPLE3",
        "company_name": "Example Company 3",
        "override_type": "ADJUST_WEIGHT",
        "model_decision": "Equal-weight allocation: 5.0%",
        "human_decision": "Increase to 7.5% weight",
        "rationale": (
            "Company is the top-ranked holding by ESG composite score (94/100) and has the "
            "highest EU Taxonomy eligibility in the portfolio (68%). However, the optimiser "
            "capped all weights at 5% by default. We apply a manual increase to 7.5% because "
            "the stock anchors our climate theme and the concentration risk is acceptable "
            "given its diversified revenue base across 8 European markets."
        ),
        "evidence": "Internal scoring table; EU Taxonomy filing 2024",
        "approved_by": "Captain + Data Engineer"
    },
    # Add more overrides below — copy one of the blocks above and edit it
]

overrides_df = pd.DataFrame(OVERRIDES)
print(f"{len(overrides_df)} override decisions recorded.")
print(f"\nSummary by type:")
if not overrides_df.empty:
    print(overrides_df["override_type"].value_counts().to_string())

3 override decisions recorded.

Summary by type:
override_type
ADD              1
REMOVE           1
ADJUST_WEIGHT    1


## Step 3 — Display the override log

This is the table you reference in Q&A when asked to justify any holding.

In [3]:
print("HUMAN OVERRIDE DECISION LOG")
print("=" * 70)

for i, row in overrides_df.iterrows():
    print(f"\n  Override #{i+1}")
    print(f"  {'─'*50}")
    print(f"  Company:         {row['company_name']} ({row['ticker']})")
    print(f"  Type:            {row['override_type']}")
    print(f"  Date:            {row['date']}")
    print(f"  Decided by:      {row['decided_by']}")
    print(f"  Approved by:     {row['approved_by']}")
    print(f"  Model said:      {row['model_decision']}")
    print(f"  Team decided:    {row['human_decision']}")
    print(f"  Rationale:")
    import textwrap
    for line in textwrap.wrap(row['rationale'], 60):
        print(f"    {line}")
    print(f"  Evidence:        {row['evidence']}")

print(f"\n{'='*70}")
print(f"Total overrides: {len(overrides_df)} (requirement: >= 3)")
if len(overrides_df) >= 3:
    print("✓ Minimum override requirement met.")
else:
    print("! Add more override entries above. Minimum 3 required for Q&A.")

HUMAN OVERRIDE DECISION LOG

  Override #1
  ──────────────────────────────────────────────────
  Company:         Example Company 1 (EXAMPLE1)
  Type:            ADD
  Date:            2026-05-06
  Decided by:      Captain
  Approved by:     Team consensus
  Model said:      Ranked 24th — just outside top 20
  Team decided:    Include at 4.0% weight
  Rationale:
    Recent CSRD disclosure (Feb 2025) shows significant
    improvement in Scope 3 reporting not yet reflected in the
    ESG dataset vintage. Company filed SBTi target in March
    2025. Including this position improves sector
    diversification into Healthcare.
  Evidence:        Sustainability Report 2024, p.22; SBTi registry confirmation email 15 Mar 2025

  Override #2
  ──────────────────────────────────────────────────
  Company:         Example Company 2 (EXAMPLE2)
  Type:            REMOVE
  Date:            2026-05-06
  Decided by:      ESG Specialist
  Approved by:     Team consensus
  Model said:      Ranked 8th —

## Step 4 — AI Use Statement

Required in the report appendix. This cell generates the standard AI Use Statement for the assignment.

In [4]:
# Load mandate for fund name
mandate_path = "../outputs/scores/mandate.json"
if os.path.exists(mandate_path):
    with open(mandate_path) as f:
        mandate = json.load(f)
    fund_name = mandate.get("fund_name", "ESADE Sustainable European Equity Fund")
else:
    fund_name = "ESADE Sustainable European Equity Fund"

ai_statement = f"""
AI USE STATEMENT
================

Fund: {fund_name}
Date: {TODAY}

1. TOOLS USED
   - Claude (Anthropic) — mandate drafting, ESG analysis, greenwashing detection
     (8-Test forensic analysis), document intelligence (RAG extraction from PDFs),
     and natural language generation for report sections
   - Python / Jupyter Notebooks — quantitative data processing, ESG scoring,
     financial metrics calculation, portfolio optimisation, chart generation
   - yfinance — market price data retrieval
   - n8n.cloud — pipeline orchestration connecting agents

2. WHAT AI DID
   - Generated all Python code in notebooks (team verified outputs, not code)
   - Extracted structured data from PDF sustainability reports with citations
   - Applied the 8-dimension greenwashing test to each company's sustainability claims
   - Drafted and revised report sections based on team-provided data
   - Assisted in interpreting regulatory frameworks (EU Taxonomy, SFDR, CSRD/ESRS)

3. WHAT HUMANS DID
   - Defined all investment mandate parameters and scoring weights
   - Verified 30% random sample of AI-extracted data against source PDFs
   - Verified 100% of exclusion and watchlist decisions against source PDFs
   - Exercised {len(overrides_df)} documented override decisions (see override log above)
   - Reviewed and approved all portfolio construction decisions
   - Validated all financial calculations against source data

4. HALLUCINATION CONTROLS
   - All Claude Projects outputs marked MISSING where information was absent
   - No data was invented, inferred, or synthesised without explicit disclosure
   - Every AI-extracted data point cites verbatim source quote and page number
   - AI-estimated values are classified as 'estimated' in the data dictionary
   - ESG ratings treated as indicators, not objective truth

5. LIMITATIONS
   - Biodiversity scores use sector-level proxies (ENCORE / WRI Aqueduct)
     due to absence of company-level TNFD disclosures
   - EU Taxonomy alignment data is sparse; eligibility used as proxy
   - Market data sourced from Yahoo Finance (adjusted prices, potential delays)
   - AI extraction quality depends on PDF text layer quality

This portfolio is an academic prototype produced for ESADE MSc Finance and does
not constitute financial advice or a regulated investment product.
"""

print(ai_statement)


AI USE STATEMENT

Fund: ESADE Sustainable European Equity Fund
Date: 2026-05-06

1. TOOLS USED
   - Claude (Anthropic) — mandate drafting, ESG analysis, greenwashing detection
     (8-Test forensic analysis), document intelligence (RAG extraction from PDFs),
     and natural language generation for report sections
   - Python / Jupyter Notebooks — quantitative data processing, ESG scoring,
     financial metrics calculation, portfolio optimisation, chart generation
   - yfinance — market price data retrieval
   - n8n.cloud — pipeline orchestration connecting agents

2. WHAT AI DID
   - Generated all Python code in notebooks (team verified outputs, not code)
   - Extracted structured data from PDF sustainability reports with citations
   - Applied the 8-dimension greenwashing test to each company's sustainability claims
   - Drafted and revised report sections based on team-provided data
   - Assisted in interpreting regulatory frameworks (EU Taxonomy, SFDR, CSRD/ESRS)

3. WHAT HUMANS 

## Step 5 — Save all outputs

In [5]:
os.makedirs("../outputs/scores", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)

# Save override log
override_path = f"../outputs/scores/human_overrides_{TODAY}.csv"
overrides_df.to_csv(override_path, index=False)
print(f"Override log saved:   {override_path}")

# Save AI Use Statement as text file
statement_path = f"../outputs/reports/ai_use_statement_{TODAY}.txt"
with open(statement_path, "w", encoding="utf-8") as f:
    f.write(ai_statement)
print(f"AI Use Statement saved: {statement_path}")

print("\nHuman review layer complete.")

Override log saved:   ../outputs/scores/human_overrides_2026-05-06.csv
AI Use Statement saved: ../outputs/reports/ai_use_statement_2026-05-06.txt

Human review layer complete.


## ✅ Notebook complete

You now have:
- A **documented override log** with rationale for every deviation from the quantitative model
- An **AI Use Statement** ready to paste into the report appendix
- Evidence that the AI is a tool supporting human judgement, not replacing it

**For Q&A on human oversight:**  
- Open this notebook and point to the override log table
- Be ready to explain Override #1 in detail (pick the most interesting one)
- Emphasise: the model creates a default ranking; humans make the final decision

**Next:** Open `05_reporting.ipynb` to generate charts and the portfolio factsheet.